# Silver Layer

## Step 1: Read Bronze Tables

In [0]:
bronze_learners_df = spark.table("learntrack_lms_analytics.default.bronze_learners")

bronze_courses_df = spark.table("learntrack_lms_analytics.default.bronze_courses")

bronze_enrolment_df = spark.table("learntrack_lms_analytics.default.bronze_enrolment_activity")

In [0]:
(bronze_learners_df).show()


+----------+----------------+--------------------+------------+----------+-----------------+-----------------+
|learner_id|    learner_name|               email|phone_number|      city|registration_date|subscription_type|
+----------+----------------+--------------------+------------+----------+-----------------+-----------------+
|   LRN0001|   Ananya Sharma|ananya.sharma.1@g...|  9433218196|    Mumbai|       2022-04-06|             Free|
|   LRN0002|   Sachin Pillai|sachin.pillai.2@g...|  9083863794|    Indore|       2022-06-13|          Premium|
|   LRN0003|   Suresh Mishra|suresh.mishra.3@r...|  9235116155|   Chennai|       2022-02-14|          Premium|
|   LRN0004|     Vijay Kumar|vijay.kumar.4@gma...|  9618495931|     Surat|       2022-08-22|          Premium|
|   LRN0005|Priya Chatterjee|priya.chatterjee....|  9316475255|     Surat|       2022-10-01|          Premium|
|   LRN0006|    Karan Pillai|karan.pillai.6@ou...|  9283276483|    Bhopal|       2022-02-27|             Free|
|

In [0]:

(bronze_courses_df).show()



+---------+--------------------+------------------+-------------+---------------+--------------+----------------+---------+
|course_id|        course_title|          category|instructor_id|instructor_name|duration_hours|difficulty_level|price_inr|
+---------+--------------------+------------------+-------------+---------------+--------------+----------------+---------+
|   CRS001|Python for Data S...|      Data Science|       INS009|   Suresh Gupta|            15|    Intermediate|     2986|
|   CRS002|Machine Learning ...|           AI & ML|       INS006|    Priya Singh|            10|        Beginner|      223|
|   CRS003|Deep Learning wit...|           AI & ML|       INS004|     Sunita Rao|            10|        Advanced|    13612|
|   CRS004|React.js Complete...|   Web Development|       INS014|   Pooja Mishra|            30|        Advanced|     6867|
|   CRS005|Node.js Backend D...|   Web Development|       INS007|    Vikas Mehta|            20|        Beginner|      119|
|   CRS0

In [0]:
(bronze_enrolment_df).show()

+------------+----------+---------+----------+------------------------+----------------------+-----------+------------+------------------+----------------+--------+---------------+------------------+
|enrolment_id|learner_id|course_id|enrol_date|expected_completion_date|actual_completion_date|     status|progress_pct|last_activity_date|assessment_score|attempts|feedback_rating|certificate_issued|
+------------+----------+---------+----------+------------------------+----------------------+-----------+------------+------------------+----------------+--------+---------------+------------------+
|    ENR00460|   LRN0376|   CRS001|2024-01-13|              2024-02-12|            2024-02-21|  Completed|         100|        2024-02-21|           79.22|       1|              2|               Yes|
|    ENR00681|   LRN0486|   CRS055|2024-01-16|              2024-01-26|            2024-01-19|  Completed|         100|        2024-01-19|           56.62|       1|              3|                No|


## Step 2: Remove Duplicate Enrolments

In [0]:
bronze_enrolment_df.count()

2000

In [0]:
silver_enrolment_df = bronze_enrolment_df.dropDuplicates(["enrolment_id"])

In [0]:
print("Before :", bronze_enrolment_df.count())


Before : 2000


In [0]:
print("After  :", silver_enrolment_df.count())

After  : 1990


## Step 3: Check Missing Values

In [0]:
from pyspark.sql.functions import col, sum, when

display(silver_enrolment_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in silver_enrolment_df.columns
]))

enrolment_id,learner_id,course_id,enrol_date,expected_completion_date,actual_completion_date,status,progress_pct,last_activity_date,assessment_score,attempts,feedback_rating,certificate_issued
0,0,0,0,0,1172,0,0,329,1172,0,1076,0


In [0]:
bronze_courses_df.select([
    sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in bronze_courses_df.columns
]).show()

+---------+------------+--------+-------------+---------------+--------------+----------------+---------+
|course_id|course_title|category|instructor_id|instructor_name|duration_hours|difficulty_level|price_inr|
+---------+------------+--------+-------------+---------------+--------------+----------------+---------+
|        0|           0|       0|            0|              6|             0|               0|        0|
+---------+------------+--------+-------------+---------------+--------------+----------------+---------+



## Step 4: Handle Missing instructor_name

In [0]:
bronze_courses_df.filter(
    col("instructor_name").isNull()
).show()

+---------+--------------------+---------------+-------------+---------------+--------------+----------------+---------+
|course_id|        course_title|       category|instructor_id|instructor_name|duration_hours|difficulty_level|price_inr|
+---------+--------------------+---------------+-------------+---------------+--------------+----------------+---------+
|   CRS006|  HTML & CSS Mastery|Web Development|       INS015|           NULL|             5|    Intermediate|     2276|
|   CRS016|Ethical Hacking B...|  Cybersecurity|       INS010|           NULL|            10|        Beginner|      686|
|   CRS020|SQL & Database De...|   Data Science|       INS013|           NULL|             5|        Advanced|     8138|
|   CRS038|Reinforcement Lea...|        AI & ML|       INS010|           NULL|            15|        Advanced|    10834|
|   CRS042|Operations Manage...|       Business|       INS001|           NULL|             5|        Advanced|     6117|
|   CRS060| Bayesian Statistics|

In [0]:
silver_courses_df = bronze_courses_df.fillna({
    "instructor_name": "Unknown"
})

## Step 5: Convert Date Columns

In [0]:
from pyspark.sql.functions import to_date
silver_learners_df = bronze_learners_df.withColumn(
    "registration_date",
    to_date("registration_date")
)

### Enrolment dates:

In [0]:
silver_enrolment_df = silver_enrolment_df \
.withColumn("enrol_date", to_date("enrol_date")) \
.withColumn("expected_completion_date", to_date("expected_completion_date")) \
.withColumn("actual_completion_date", to_date("actual_completion_date")) \
.withColumn("last_activity_date", to_date("last_activity_date"))

## Step 6: Create Derived Columns

In [0]:
from pyspark.sql.functions import datediff
silver_enrolment_df = silver_enrolment_df.withColumn(
    "learning_duration_days",
    datediff(
        col("actual_completion_date"),
        col("enrol_date")
    )
)

## Completion Delay

In [0]:
silver_enrolment_df = silver_enrolment_df.withColumn(
    "completion_delay_days",
    datediff(
        col("actual_completion_date"),
        col("expected_completion_date")
    )
)

## Step 7: Join the Tables

In [0]:
silver_df = (
    silver_enrolment_df.alias("e")
    .join(
        silver_learners_df.alias("l"),
        on="learner_id",
        how="left"
    )
    .join(
        silver_courses_df.alias("c"),
        on="course_id",
        how="left"
    )
)

## Step 8: Verify

In [0]:
silver_df.show(10)

+---------+----------+------------+----------+------------------------+----------------------+-----------+------------+------------------+----------------+--------+---------------+------------------+----------------------+---------------------+-----------------+--------------------+------------+----------+-----------------+-----------------+--------------------+------------------+-------------+---------------+--------------+----------------+---------+
|course_id|learner_id|enrolment_id|enrol_date|expected_completion_date|actual_completion_date|     status|progress_pct|last_activity_date|assessment_score|attempts|feedback_rating|certificate_issued|learning_duration_days|completion_delay_days|     learner_name|               email|phone_number|      city|registration_date|subscription_type|        course_title|          category|instructor_id|instructor_name|duration_hours|difficulty_level|price_inr|
+---------+----------+------------+----------+------------------------+-----------------

In [0]:
silver_df.printSchema()

root
 |-- course_id: string (nullable = true)
 |-- learner_id: string (nullable = true)
 |-- enrolment_id: string (nullable = true)
 |-- enrol_date: date (nullable = true)
 |-- expected_completion_date: date (nullable = true)
 |-- actual_completion_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- progress_pct: integer (nullable = true)
 |-- last_activity_date: date (nullable = true)
 |-- assessment_score: double (nullable = true)
 |-- attempts: integer (nullable = true)
 |-- feedback_rating: integer (nullable = true)
 |-- certificate_issued: string (nullable = true)
 |-- learning_duration_days: integer (nullable = true)
 |-- completion_delay_days: integer (nullable = true)
 |-- learner_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: long (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- course_title: string (nullable = tr

In [0]:
silver_df.count()

1990

## Step 9: Save Silver Tables

In [0]:
silver_learners_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("learntrack_lms_analytics.default.silver_learners")

In [0]:
silver_courses_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("learntrack_lms_analytics.default.silver_courses")

In [0]:
silver_enrolment_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("learntrack_lms_analytics.default.silver_enrolment_activity")

In [0]:
silver_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("learntrack_lms_analytics.default.silver_lms")